In [3]:
import json

config_path = "/Users/tongwenchao/ml-projects/lora-finetune/mistral-7b-sentiment-merged/config.json"

with open(config_path) as f:
    config = json.load(f)

# 看看 quantization_config 长什么样
print(json.dumps(config.get("quantization_config", "NOT FOUND"), indent=2))

{
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
import torch

model_path = "/Users/tongwenchao/ml-projects/lora-finetune/mistral-7b-sentiment-merged"

tokenizer = AutoTokenizer.from_pretrained(model_path)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="mps",
)
model.eval()
print("Done! device:", next(model.parameters()).device)

/opt/anaconda3/envs/mlenv/lib/python3.11/site-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Done! device: mps:0


In [5]:
id2label = {0: "Bearish", 1: "Bullish", 2: "Neutral"}

texts = [
    "Federal Reserve holds interest rates steady",
    "Company reports massive losses this quarter",
    "$AAPL hits all-time high after earnings beat",
    "No, The Fed Won't Save The Market",
    "$KTOS awarded $39M contract",
]

device = next(model.parameters()).device
inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits
preds = logits.argmax(dim=-1)

for text, pred in zip(texts, preds):
    print(f"{id2label[pred.item()]:10s} | {text}")

/opt/anaconda3/envs/mlenv/lib/python3.11/site-packages/bitsandbytes/backends/default/ops.py:204: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/opt/anaconda3/envs/mlenv/lib/python3.11/site-packages/bitsandbytes/backends/default/ops.py:314: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Neutral    | Federal Reserve holds interest rates steady
Bearish    | Company reports massive losses this quarter
Bullish    | $AAPL hits all-time high after earnings beat
Neutral    | No, The Fed Won't Save The Market
Neutral    | $KTOS awarded $39M contract
